In [1]:
#!pip install xgboost

In [1]:
import pandas as pd
import numpy as np

In [2]:
from sklearn.model_selection import train_test_split, cross_val_score, cross_val_predict
from sklearn.metrics import confusion_matrix, f1_score

In [3]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.svm import SVC

In [4]:
from xgboost import XGBClassifier

In [5]:
from sklearn.ensemble import VotingClassifier, GradientBoostingClassifier
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier
from sklearn.ensemble import ExtraTreesClassifier, AdaBoostClassifier

In [6]:
df = pd.read_csv ('titanik.csv')
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [7]:
df.drop(columns = ['PassengerId','Name','Ticket','Cabin'], inplace = True)

In [8]:
df['Age'] = df['Age'].fillna(28)
df['Embarked'] = df['Embarked'].fillna('U')
df['Sex'] = np.where(df['Sex'] == 'male', 1,0)
df['Embarked'] = df['Embarked'].map({'U':0, 'S':1,'C':2, 'Q':3})

In [9]:
array = df.values
X = array[:,1::]
y = array[:,0]

In [10]:
X_train,X_test,y_train,y_test = train_test_split (X,y, test_size = 0.2, random_state = 17)

### Бэггинг

In [11]:
model_RF = RandomForestClassifier(n_estimators = 100, random_state = 7)
model_RF.fit(X_train,y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [12]:
y_pred = model_RF.predict(X_test)
f1_score(y_test, y_pred)

0.7310344827586207

In [13]:
cart = DecisionTreeClassifier()
model_bagged = BaggingClassifier(estimator = cart, n_estimators = 100, random_state = 7)
model_bagged.fit(X_train, y_train)

,estimator,DecisionTreeClassifier()
,n_estimators,100
,max_samples,1.0
,max_features,1.0
,bootstrap,True
,bootstrap_features,False
,oob_score,False
,warm_start,False
,n_jobs,None
,random_state,7
,verbose,0


In [14]:
y_pred = model_bagged.predict(X_test)
f1_score(y_test,y_pred)

0.7464788732394366

In [15]:
model_ET = ExtraTreesClassifier (n_estimators = 100, random_state = 7)
model_ET.fit(X_train, y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,False
,oob_score,False


In [16]:
y_pred = model_ET.predict(X_test)
f1_score(y_test,y_pred)

0.7123287671232876

In [17]:
estimators = []
estimators.append(('knn',KNeighborsClassifier()))
estimators.append(('logreg',LogisticRegression()))
estimators.append(('gb',GaussianNB()))
estimators.append(('cart',DecisionTreeClassifier()))
estimators.append(('ld', LinearDiscriminantAnalysis()))

In [18]:
vc  = VotingClassifier (estimators)
vc.fit(X_train, y_train)

,estimators,"[('knn', ...), ('logreg', ...), ...]"
,voting,'hard'
,weights,None
,n_jobs,None
,flatten_transform,True
,verbose,False
,n_neighbors,5
,weights,'uniform'
,algorithm,'auto'
,leaf_size,30
,p,2


In [19]:
y_pred = vc.predict(X_test)
f1_score(y_test,y_pred)

0.7153284671532847

### 2. Бустинг

In [21]:
#Адаптивный бустинг
model_AB = AdaBoostClassifier(n_estimators = 30, random_state = 7)
model_AB.fit (X_train,y_train)

,estimator,None
,n_estimators,30
,learning_rate,1.0
,algorithm,'deprecated'
,random_state,7


In [22]:
y_pred = model_AB.predict (X_test)
f1_score (y_test,y_pred)

0.7246376811594203

In [29]:
#Градиентный бустинг
model_GB = GradientBoostingClassifier(n_estimators = 300, random_state = 7)
model_GB.fit (X_train,y_train)

,loss,'log_loss'
,learning_rate,0.1
,n_estimators,300
,subsample,1.0
,criterion,'friedman_mse'
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_depth,3
,min_impurity_decrease,0.0
,init,None


In [30]:
y_pred = model_GB.predict (X_test)
f1_score (y_test,y_pred)

0.7567567567567568

In [35]:
#XGBoost
model_XGB = XGBClassifier (n_estimators = 100, learning_rate = 0.3, seed = 7)
model_XGB.fit (X_train, y_train)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [36]:
y_pred = model_XGB.predict(X_test)
f1_score (y_test,y_pred)

0.7777777777777778

### Стекинг

Блендинг

In [38]:
#1. Делим обучающие данные на треин и тест
train,valid,train_true,valid_true = train_test_split (X_train, y_train,
                                                     train_size = 0.5,
                                                     random_state = 0)

In [52]:
#2. Обучаем базовые алгоритмы
models = []
knn = KNeighborsClassifier(n_neighbors = 3)
knn_model = knn.fit(train,train_true)
models.append(knn_model)

lr = LogisticRegression (random_state = 17)
lr_model = lr.fit(train,train_true)
models.append(lr_model )

cart = DecisionTreeClassifier(max_leaf_nodes = 4, random_state = 17)
cart_model = cart.fit(train, train_true)
models.append(cart_model)

svc = SVC(random_state = 17)
svc_model = svc.fit (train, train_true)
models.append(svc_model)

/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [53]:
models

[KNeighborsClassifier(n_neighbors=3),
 LogisticRegression(random_state=17),
 DecisionTreeClassifier(max_leaf_nodes=4, random_state=17),
 SVC(random_state=17)]

In [54]:
#3. Получаем предсказания и заполняем метаматрицу
meta_mtrx = np.empty((valid.shape[0], len(models)))

for i, model in enumerate(models):
    meta_mtrx[:, i] = model.predict(valid)
    predicted = model.predict(X_test)
    print (f'{i} f1_score: {f1_score(y_test,predicted)}') 

0 f1_score: 0.5384615384615384
1 f1_score: 0.7142857142857143
2 f1_score: 0.6608695652173913
3 f1_score: 0.45871559633027525


In [55]:
#4. Обучим метаалгоритм
meta = XGBClassifier (n_estimators = 40)
meta_model = meta.fit(meta_mtrx, valid_true)
meta_mtrx_test = np.empty((X_test.shape[0], len(models)))

for i, model in enumerate (models):
    meta_mtrx_test[:,i] = model.predict(X_test)

meta_predict = meta.predict(meta_mtrx_test)
print (f'f1_score: {f1_score(y_test, meta_predict)}')

f1_score: 0.6984126984126984


Классический стекинг

In [57]:
# 1. Определяем базовые алгоритмы
knn = KNeighborsClassifier()
lr = LogisticRegression()
cart = DecisionTreeClassifier()
nb = GaussianNB()
ld = LinearDiscriminantAnalysis()
models = [knn,lr,cart,nb,ld]

In [59]:
#2. Определяем метаалгоритм и метаматрицу
meta_alg = XGBClassifier(n_estimators = 40)
meta_mtrx = np.empty((df.shape[0], len(models)))

In [60]:
df.shape[0]

891

In [63]:
#3.Заполняем метаматрицу
for i, model in enumerate (models):
    meta_mtrx[:,i] = cross_val_predict(model, X,y,cv=5,method = 'predict')
    base_alghorithm = model.fit(X_train,y_train)
    predicted = model.predict(X_test)
    print (f'{i} f1_score: {f1_score(y_test,predicted)}')

0 f1_score: 0.5511811023622047


/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.or

1 f1_score: 0.6944444444444444
2 f1_score: 0.7432432432432432
3 f1_score: 0.7183098591549296
4 f1_score: 0.6811594202898551


In [66]:
# 3. Обуаем метамодель
meta_model = meta_alg.fit(meta_mtrx, y)
meta_mtrx_test = np.empty((df.shape[0], len(models)))
for i, model in enumerate(models):
    meta_mtrx_test [:, i] = model.predict(X)

In [67]:
meta_predict = meta_alg.predict(meta_mtrx_test)
f1_score(y, meta_predict)

0.8061068702290076